# Warm-up Exercises
#### Theory-Informed Spectral Methods for Cross-Sectional Asset Pricing 
#### Replication of Kelly, Malamud, Zhou (2024)

This notebook contains the 3 warm-up exercises from the project brief, covering:
- The discrete Green's function
- The spectral structure of the second-difference matrix
- Two underlying identities of ridge regression

## Exercise 3.2 — The Second-Difference Matrix and its Inverse

The brief defines the second-difference matrix $A \in \mathbb{R}^{n \times n}$ with 
$n = 200$ and $h = \frac{1}{n+1}$:

$$A = \frac{1}{h^2} \begin{pmatrix} 2 & -1 & & \\ -1 & 2 & -1 & \\ & \ddots & \ddots & \ddots \\ & & -1 & 2 \end{pmatrix}$$

This discretises the differential operator $-d^2/dx^2$ on $[0,1]$ with boundary 
conditions $u(0) = u(1) = 0$, in the sense that $(Au)_i \approx -u''(x_i)$.

The brief gives the closed-form inverse:

$$(A^{-1})_{ij} = h \cdot \min(x_i, x_j)(1 - \max(x_i, x_j))$$

where $x_i = ih$ are the grid points. We verify this numerically and plot row 
$i = 100$ of $A^{-1}$, which the brief describes as a tent function — the response 
to a point impulse at $x_{100}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#Building matrix A with the diagonal structure 
array_1 = np.full((200), 2)
array_2 = np.full((199), -1)
h = 1 / 201
A = (1 / (h ** 2)) * (np.diag(array_1, k = 0) + np.diag(array_2, k = 1) + np.diag(array_2, k = -1))

#Inverting A and extracting the 100th row of the inverse 
A_inv = np.linalg.inv(A)
row_100 = A_inv[99, :]

#Computing 100th row of inverse using formula for all (i,j)
grid_points = np.arange(1, 201) * h
xi, xj = np.meshgrid(grid_points, grid_points)
formula_inverse_A = h * np.minimum(xi, xj) * (1 - np.maximum(xi, xj))
row_100_formula = formula_inverse_A[99, :]

#Plotting results 
plt.plot(grid_points, row_100, linestyle = "--")
plt.xlabel("x")
plt.ylabel("A inverse row 100")
plt.title("Row 100 of Inverse of A")
plt.show()

The plot shows the expected tent function, peaking at $x_{100} \approx 0.498$ and 
falling linearly to zero at both ends of $[0,1]$. This is the discrete Green's 
function: $(A^{-1})_{ij}$ measuring the response of the system at position $i$ to 
a unit impulse at position $j$. As $n \to \infty$ this converges to the continuous 
Green's function $G(x,y) = \min(x,y)(1 - \max(x,y))$.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(A)

for i in range(4):
    plt.plot(grid_points, eigenvectors[:, i], label = f"eigenvector {i + 1}")
plt.title("First Four Eigenvectors of A")
plt.legend(loc = "upper right", fontsize = 8)
plt.show()

## Exercise 3.4 — Eigenvectors of A

The brief asks us to compute the eigenvalues and eigenvectors of $A$ and plot the 
first four eigenvectors. Since $A$ is symmetric and real, we use `np.linalg.eigh` which 
guarantees real outputs (without complex outputs due to floating point errors) and returns eigenvalues in ascending order.

The brief predicts that the eigenfunctions of $-d^2/dx^2$ on $(0,1)$ with these 
boundary conditions are $\sin(k\pi x)$ for $k = 1, 2, 3, \ldots$ — so the discrete 
eigenvectors should approximate sine waves of increasing frequency. The graph of these eigenvectors confirms this.

The four eigenvectors approximate $\sin(\pi x)$, $\sin(2\pi x)$, $\sin(3\pi x)$ 
and $\sin(4\pi x)$ as predicted. This confirms that the discrete and continuous 
problems share their spectral structure - the basis of the research programme, which 
asks whether eigenfunctions of the pricing operator can replace random Fourier 
features as a basis for return prediction.

## Exercise 4.1 — Two Identities for Ridge Regression

Ridge regression solves:

$$\hat{\beta}(\lambda) = \arg\min_{\beta} \|r - X\beta\|^2 + \lambda\|\beta\|^2 
= (X^TX + \lambda I_P)^{-1}X^Tr, \quad \lambda > 0$$

where $X \in \mathbb{R}^{T \times P}$ and $r \in \mathbb{R}^T$. 
The exercise asks us to prove 2 identities. The first has a crucial computational consequence: when $P \gg T$ 
(as in KMZ where $P$ reaches thousands and $T = 12$), the right-hand side of 
identity 1 inverts a $T \times T$ matrix rather than a $P \times P$ matrix, 
making the computation feasible.

### Part 1

**Claim:**

$$(X^TX + \lambda I_P)^{-1}X^T = X^T(XX^T + \lambda I_T)^{-1}$$

**Proof.** Multiply both sides on the left by $(X^TX + \lambda I_P)$ and on the 
right by $(XX^T + \lambda I_T)$. The left side simplifies as:

$$(X^TX + \lambda I_P)(X^TX + \lambda I_P)^{-1} X^T (XX^T + \lambda I_T) 
= X^T(XX^T + \lambda I_T)$$

The right side simplifies as:

$$(X^TX + \lambda I_P) X^T (XX^T + \lambda I_T)^{-1}(XX^T + \lambda I_T) 
= (X^TX + \lambda I_P)X^T$$

So it suffices to show $X^T(XX^T + \lambda I_T) = (X^TX + \lambda I_P)X^T$. 
Expanding both sides:

$$\text{LHS} = X^TXX^T + \lambda X^T I_T = X^TXX^T + \lambda X^T$$

$$\text{RHS} = X^TXX^T + \lambda I_P X^T = X^TXX^T + \lambda X^T$$

Thus, both sides are equal, completing the proof. $\blacksquare$

**Computational consequence.** The identity lets us replace the inversion of a 
$P \times P$ matrix with the inversion of a $T \times T$ matrix. When $T = 12$ 
and $P = 12{,}000$, this is the difference between inverting a $12 \times 12$ 
matrix and a $12{,}000 \times 12{,}000$ matrix.

### Part 2

**Claim:** As $\lambda \downarrow 0$, $\hat{\beta}(\lambda) \to X^+r$, the 
minimum-norm least-squares solution given by the pseudoinverse.

**Proof.** Write $X = U\Sigma V^T$ (SVD), where:
- $U \in \mathbb{R}^{T \times T}$, columns $u_1,\ldots,u_T$ orthonormal: $U^TU = I_T$
- $\Sigma \in \mathbb{R}^{T \times P}$, diagonal entries $\sigma_1 \geq \cdots \geq \sigma_T \geq 0$
- $V \in \mathbb{R}^{P \times P}$, columns $v_1,\ldots,v_P$ orthonormal: $V^TV = I_P$

**Step 1.** Compute $X^TX$:

$$X^TX = V\Sigma^T U^T U \Sigma V^T = V\Sigma^T\Sigma V^T$$

since $U^TU = I_T$. Here $\Sigma^T\Sigma \in \mathbb{R}^{P \times P}$ is diagonal 
with $(i,i)$ entry $\sigma_i^2$ for $i \leq T$ and $0$ for $i > T$.

**Step 2.** Factor $X^TX + \lambda I_P$:

$$X^TX + \lambda I_P = V\Sigma^T\Sigma V^T + \lambda VV^T 
= V(\Sigma^T\Sigma + \lambda I_P)V^T$$

using $I_P = VV^T$.

**Step 3.** Invert, using $(VAV^T)^{-1} = VA^{-1}V^T$ for orthogonal $V$:

$$(X^TX + \lambda I_P)^{-1} = V(\Sigma^T\Sigma + \lambda I_P)^{-1}V^T$$

Since $\Sigma^T\Sigma + \lambda I_P$ is diagonal, its inverse has entries 
$\frac{1}{\sigma_i^2 + \lambda}$ for $i \leq T$ and $\frac{1}{\lambda}$ for $i > T$.

**Step 4.** Substitute into the ridge formula using identity 1 
(the $T \times T$ form is cheaper):

$$\hat{\beta}(\lambda) = (X^TX + \lambda I_P)^{-1}X^Tr 
= V(\Sigma^T\Sigma + \lambda I_P)^{-1}V^T \cdot V\Sigma^T U^T r$$

Since $V^TV = I_P$:

$$= V(\Sigma^T\Sigma + \lambda I_P)^{-1}\Sigma^T U^T r$$

The $(i,i)$ entry of $(\Sigma^T\Sigma + \lambda I_P)^{-1}\Sigma^T$ is 
$\frac{\sigma_i}{\sigma_i^2 + \lambda}$ for $i \leq T$ and $0$ for $i > T$, so:

$$\hat{\beta}(\lambda) = V \cdot D(\lambda) \cdot U^T r$$

where $D(\lambda) \in \mathbb{R}^{P \times T}$ has $(i,i)$ entry 
$\frac{\sigma_i}{\sigma_i^2+\lambda}$.

**Step 5.** Take $\lambda \downarrow 0$. Each diagonal entry satisfies:

$$\frac{\sigma_i}{\sigma_i^2 + \lambda} \xrightarrow{\lambda \to 0} 
\begin{cases} \dfrac{1}{\sigma_i} & \text{if } \sigma_i > 0 \\ 0 & \text{if } \sigma_i = 0 \end{cases}$$

So $D(\lambda) \to \Sigma^+$, the pseudoinverse of $\Sigma$, giving:

$$\hat{\beta}(\lambda) \xrightarrow{\lambda \downarrow 0} V\Sigma^+ U^T r = X^+ r$$

where $X^+ = V\Sigma^+U^T$ is the pseudoinverse of $X$. This is the minimum-norm 
least-squares solution — among all $\beta$ minimising $\|r - X\beta\|^2$, it is 
the one with smallest $\|\beta\|^2$. $\blacksquare$

## KMZ Empirical Design Notes

The following summarises the exact empirical design choices of Kelly, Malamud 
and Zhou (2024), "The Virtue of Complexity in Return Prediction", *Journal of 
Finance* 79(1), pp. 459–506, as required by the project brief (Step 0). Section 
and page references are to the published paper.

### Forecast target (Section V.A, p. 487)
- Monthly **excess return of the CRSP value-weighted index**.
- Returns are **volatility-standardised** using a backward-looking, trailing 
  12-month return standard deviation, computed from the *uncentered* second 
  moment (footnote 34). This preserves the out-of-sample property.

### Predictors (Section V.A, p. 487 and footnote 33)
- The **15 monthly predictor variables** from Goyal and Welch (2008), 1926–2020. 
  Mnemonics: dfy, infl, svar, de, lty, tms, tbl, dfr, dp, dy, ltr, ep, b/m, ntis, 
  plus **one lag of the market return**.
- Predictors are standardised using an **expanding-window** historical standard 
  deviation; returns use a rolling 12-month window (predictors are more 
  persistent, so an expanding window is used).
- **36 months of data required** for initial standardisation stability, so the 
  effective sample begins in **1930**.
- Date convention for inflation follows Goyal, Welch, and Zafirov (2023): 
  month-$t$ price data is treated as part of the time-$t$ information set 
  (footnote 33).

### Feature construction — Random Fourier Features (Section V.B, pp. 487–488)
- RFF map (equation 20): for the $15 \times 1$ predictor vector $G_t$,

$$S_{i,t} = \left(\sin(\gamma\, \omega_i' G_t),\ \cos(\gamma\, \omega_i' G_t)\right), 
\qquad \omega_i \overset{iid}{\sim} \mathcal{N}(0, I)$$

- **Bandwidth $\gamma = 2$** in the main analysis (footnote 38); robustness 
  checks use $\gamma = 0.5$ and $\gamma = 1$ (Section V.F).
- Features are drawn once and **fixed**; models of increasing $P$ are **nested** — 
  the first $P_1$ features of a $P_2$-model ($P_1 < P_2$) form the $P_1$-model 
  (Section V.C, step ii).
- Training-sample RFFs are volatility-standardised by their training-sample 
  standard deviations; the out-of-sample RFF $S_t$ is standardised by the same 
  training-sample values (footnote 39 — no look-ahead).

### Model and estimation (Sections II.A and V.C)
- **Ridge regression** of $R_{t+1}$ on $S_t$:

$$\hat{\beta}(z) = \left(z I + \frac{1}{T}\sum_t S_t S_t'\right)^{-1} 
\frac{1}{T}\sum_t S_t R_{t+1}$$

- **No intercept** in the regressions (footnote 35).
- **Ridgeless limit** ($z \to 0^+$) computed via the Moore–Penrose pseudoinverse 
  (Section II.A, footnote 23).

### Experimental grids (Section V.C)
- **Training windows:** $T \in \{12, 60, 120\}$ months (rolling).
- **Feature counts:** $P$ from $2$ up to **12,000**.
- **Shrinkage grid:** $\log_{10}(z) \in \{-3, -2, -1, 0, 1, 2, 3\}$.
- **Complexity** defined as $c = P/T$.
- Each configuration is repeated with **1,000 independent RFF draws**, with 
  performance statistics **averaged across repetitions** (Section V.C). 
  *Note: the project brief requires $\geq 10$ seeds with the seed-to-seed spread 
  shown, as a computationally lighter alternative.*

### Out-of-sample protocol (Section V.C, step iii)
- For each month $t$: estimate on the trailing window 
  $\{(R_t, S_{t-1}), \ldots, (R_{t-T+1}, S_{t-T})\}$, then forecast $R_{t+1}$ 
  as $\hat{\beta}' S_t$ and record the timing-strategy return 
  $\hat{\beta}' S_t \cdot R_{t+1}$.
- **Timing position:** $\pi_t = \hat{\beta}' S_t$ — proportional to the forecast 
  (equation 6). No position constraints imposed.

### Performance metrics (Sections I.B, V.C, footnote 40)
- **Out-of-sample $R^2$:**

$$R^2_{OOS} = 1 - \frac{\sum_t (r_{t+1} - \hat{r}_{t+1})^2}{\sum_t r_{t+1}^2}$$

- **Sharpe ratio** of the timing strategy, annualised by $\sqrt{12}$, using the 
  centered standard deviation in the denominator.
- Also reported: average $\|\hat{\beta}\|^2$ across training samples; strategy 
  expected return and volatility; alpha and information ratio versus a static 
  position in the volatility-standardised market (Table I).

### Headline results to replicate (Figures 7–8, Table I, pp. 490–495)
- **VoC curves:** $R^2_{OOS}$ spikes negative at $c = 1$ (the interpolation 
  boundary) and recovers beyond it (double descent); $\|\hat{\beta}\|$ spikes 
  at $c = 1$.
- Expected timing return **rises with complexity**, roughly flat beyond $c = 1$ 
  in the ridgeless case (Proposition 6, equation 19).
- Sharpe ratio **increases with complexity**, dipping near $c = 1$ at low 
  shrinkage, exceeding $\approx 0.4$ at high complexity.
- Table I benchmarks ($T = 12$): nonlinear model with $c = 1000$, $z = 10^3$ 
  achieves $R^2_{OOS} \approx 0.6\%$, $SR \approx 0.47$, $IR \approx 0.31$ vs 
  market; the ridgeless linear "kitchen sink" gives $R^2_{OOS} < -100\%$, 
  $SR \approx -0.11$.
- **Acceptance criteria (brief):** high-complexity tuned-$\lambda$ Sharpe within 
  $\pm 0.1$ of the paper's headline; $R^2_{OOS}$ of the same sign and order of 
  magnitude.